# 03 — Model Training

Trains two AutoGluon models:
- **W/D/L Classifier** — predicts match result (win / draw / loss) using `log_loss`
- **Goal Difference Regressor** — predicts `home_score - away_score` using `mean_absolute_error`

Validation: 2022 FIFA World Cup matches (2022-11-20 → 2022-12-18).

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from autogluon.tabular import TabularPredictor

import matplotlib.pyplot as plt

In [ ]:
PROCESSED_DIR = Path("../data/processed")
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(exist_ok=True)

# Load training set
df = pd.read_csv(PROCESSED_DIR / "training_set.csv", parse_dates=["date"])

# Hold out 2022 World Cup for validation
WC2022_START = pd.Timestamp("2022-11-20")
WC2022_END   = pd.Timestamp("2022-12-18")

val_mask  = (df["date"] >= WC2022_START) & (df["date"] <= WC2022_END)
train_df  = df[~val_mask].copy()
val_df    = df[val_mask].copy()

print(f"Train rows : {len(train_df):,}")
print(f"Val rows   : {len(val_df):,}  (2022 World Cup)")
print()
print("Train result distribution:")
print(train_df["result"].value_counts(normalize=True).rename("share").to_string())
print()
print("Val result distribution:")
print(val_df["result"].value_counts(normalize=True).rename("share").to_string())

In [ ]:
# Columns to exclude from features
NON_FEATURE_COLS = {"date", "home_team", "away_team", "result", "goal_diff", "sample_weight"}

feature_cols = [c for c in df.columns if c not in NON_FEATURE_COLS]

print(f"Feature columns ({len(feature_cols)}):")
for col in feature_cols:
    print(f"  {col:40s}  {str(df[col].dtype)}")

In [ ]:
clf_train = train_df[feature_cols + ["result", "sample_weight"]].copy()
clf_val   = val_df[feature_cols + ["result", "sample_weight"]].copy()

predictor_clf = TabularPredictor(
    label="result",
    eval_metric="log_loss",
    path=str(MODELS_DIR / "classifier"),
)
predictor_clf.fit(
    clf_train,
    sample_weight="sample_weight",
    presets="best_quality",
    time_limit=600,
)

print("\n=== Classifier leaderboard (validation set) ===")
leaderboard_clf = predictor_clf.leaderboard(clf_val, silent=True)
print(leaderboard_clf[["model", "score_val", "score_test"]].to_string(index=False))

In [ ]:
from sklearn.metrics import accuracy_score

clf_val_features = clf_val.drop(columns=["result", "sample_weight"])
y_true = clf_val["result"]
y_pred = predictor_clf.predict(clf_val_features)

acc = accuracy_score(y_true, y_pred)
print(f"Accuracy on 2022 WC validation set: {acc:.3f}")
print()

pred_dist   = y_pred.value_counts(normalize=True).rename("predicted")
actual_dist = y_true.value_counts(normalize=True).rename("actual")
dist_df = pd.concat([actual_dist, pred_dist], axis=1).fillna(0)
print("Result distribution — actual vs predicted:")
print(dist_df.to_string())

In [ ]:
reg_train = train_df[feature_cols + ["goal_diff", "sample_weight"]].copy()
reg_val   = val_df[feature_cols + ["goal_diff", "sample_weight"]].copy()

predictor_reg = TabularPredictor(
    label="goal_diff",
    problem_type="regression",
    eval_metric="mean_absolute_error",
    path=str(MODELS_DIR / "regressor"),
)
predictor_reg.fit(
    reg_train,
    sample_weight="sample_weight",
    presets="best_quality",
    time_limit=600,
)

print("\n=== Regressor leaderboard (validation set) ===")
leaderboard_reg = predictor_reg.leaderboard(reg_val, silent=True)
print(leaderboard_reg[["model", "score_val", "score_test"]].to_string(index=False))

In [ ]:
reg_val_features = reg_val.drop(columns=["goal_diff", "sample_weight"])
y_true_reg = reg_val["goal_diff"]
y_pred_reg = predictor_reg.predict(reg_val_features)

mae = np.mean(np.abs(y_true_reg - y_pred_reg))
print(f"MAE on 2022 WC validation set: {mae:.3f}")
print()

# Show predictions vs actuals for each WC match
comparison = val_df[["date", "home_team", "away_team", "goal_diff"]].copy().reset_index(drop=True)
comparison["pred_goal_diff"] = y_pred_reg.values
comparison["abs_error"] = (comparison["goal_diff"] - comparison["pred_goal_diff"]).abs()
comparison = comparison.sort_values("date").reset_index(drop=True)

print("2022 WC matches — actual vs predicted goal difference:")
print(comparison[["date", "home_team", "away_team", "goal_diff", "pred_goal_diff", "abs_error"]].to_string(index=False))

In [ ]:
# Feature importance — top 10 for each model
fi_clf = predictor_clf.feature_importance(clf_val).head(10)
fi_reg = predictor_reg.feature_importance(reg_val).head(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(fi_clf.index[::-1], fi_clf["importance"][::-1], color="steelblue", edgecolor="white")
axes[0].set_title("W/D/L Classifier — Top 10 Features")
axes[0].set_xlabel("Importance")

axes[1].barh(fi_reg.index[::-1], fi_reg["importance"][::-1], color="coral", edgecolor="white")
axes[1].set_title("Goal Diff Regressor — Top 10 Features")
axes[1].set_xlabel("Importance")

plt.tight_layout()
plt.show()